# Stationarity and Differencing

Most classical time series models (ARIMA, exponential smoothing, etc.) assume
or require that the data are **stationary**.  This notebook covers:

1. What stationarity means (strict vs weak)
2. Why it matters for forecasting
3. Visual rolling-statistics test
4. The **Augmented Dickey-Fuller (ADF)** test
5. The **KPSS** test
6. **Differencing** to achieve stationarity
7. Over-differencing warning

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Load datasets ---
# Airline Passengers (non-stationary: trend + seasonality + growing variance)
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# Daily Female Births (stationary)
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

# Alcohol Sales (non-stationary: trend + seasonality)
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]

print("Airline: ", airline.shape)
print("Births:  ", births.shape)
print("Alcohol: ", alcohol.shape)

---
## 1. What Is Stationarity?

### Strict Stationarity

A time series $\{y_t\}$ is **strictly stationary** if the joint probability
distribution of $(y_{t_1}, y_{t_2}, \ldots, y_{t_k})$ is identical to the
distribution of $(y_{t_1+h}, y_{t_2+h}, \ldots, y_{t_k+h})$ for all choices
of times $t_1, \ldots, t_k$ and shift $h$.  In other words, the statistical
properties of the series are completely invariant under time shifts.

This is a very strong condition and is rarely verifiable in practice.

### Weak (Covariance) Stationarity

A more practical definition.  A series is **weakly stationary** if:

1. **Constant mean:** $E[y_t] = \mu$ for all $t$
2. **Constant variance:** $\text{Var}(y_t) = \sigma^2$ for all $t$
3. **Autocovariance depends only on lag:** $\text{Cov}(y_t, y_{t-k}) = \gamma_k$ for all $t$

In practice, when we say "stationary" we almost always mean *weakly stationary*.

### Why Does Stationarity Matter?

- ARIMA, VAR, and many other models estimate **fixed parameters** ($\phi$,
  $\theta$, $\sigma^2$) from the data.  If the underlying distribution changes
  over time, those estimates are meaningless.
- The ACF and PACF assume stationarity.  A non-stationary series produces a
  slowly-decaying ACF that masks the true dependence structure.
- Forecasts from a non-stationary model can diverge or exhibit spurious
  trends.

---
## 2. Visual Test: Rolling Mean and Rolling Std

A quick visual check: if the **rolling mean** and **rolling standard
deviation** are approximately constant over time, the series is likely
stationary.

In [ ]:
def plot_rolling_stats(series, window=12, title=""):
    """Plot the series with its rolling mean and rolling std."""
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Rolling mean
    axes[0].plot(series, alpha=0.5, label="Original")
    axes[0].plot(rolling_mean, color="red", linewidth=2, label=f"Rolling Mean ({window})")
    axes[0].set_title(f"{title} — Rolling Mean")
    axes[0].legend()

    # Rolling std
    axes[1].plot(rolling_std, color="orange", linewidth=2)
    axes[1].set_title(f"{title} — Rolling Std ({window})")
    axes[1].set_ylabel("Standard Deviation")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rolling_stats(airline["Passengers"], window=12, title="Airline Passengers")

print("Rolling mean trends upward — the mean is NOT constant.")
print("Rolling std also increases — the variance is NOT constant.")
print("Verdict: clearly NON-STATIONARY.")

In [ ]:
plot_rolling_stats(births["Births"], window=30, title="Daily Female Births")

print("Rolling mean is roughly flat around 40-42.")
print("Rolling std is roughly constant around 7.")
print("Verdict: looks STATIONARY.")

In [ ]:
plot_rolling_stats(alcohol["Sales"], window=12, title="Alcohol Sales")

print("Rolling mean trends upward.")
print("Verdict: NON-STATIONARY.")

---
## 3. Augmented Dickey-Fuller (ADF) Test

The ADF test is the most widely used formal test for stationarity.

### Hypotheses

- $H_0$: The series has a **unit root** (it is non-stationary)
- $H_1$: The series is **stationary** (no unit root)

### Interpretation

- **p-value < 0.05** $\Rightarrow$ reject $H_0$ $\Rightarrow$ evidence of stationarity
- **p-value $\ge$ 0.05** $\Rightarrow$ fail to reject $H_0$ $\Rightarrow$ evidence of non-stationarity

The test is based on fitting the regression:

$$
\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t
$$

and testing $\gamma = 0$ (unit root).

In [ ]:
def adf_test(series, name=""):
    """Run the Augmented Dickey-Fuller test and print a clear summary.

    Parameters
    ----------
    series : array-like
        The time series to test.
    name : str
        Label for display purposes.

    Returns
    -------
    dict with test statistic, p-value, lags used, and critical values.
    """
    result = adfuller(series.dropna(), autolag="AIC")
    output = {
        "Test Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations": result[3],
        "Critical Values": result[4],
    }

    label = f" ({name})" if name else ""
    print(f"Augmented Dickey-Fuller Test{label}")
    print(f"  Test Statistic : {result[0]:.6f}")
    print(f"  p-value        : {result[1]:.6f}")
    print(f"  Lags Used      : {result[2]}")
    print(f"  Observations   : {result[3]}")
    for key, val in result[4].items():
        print(f"  Critical Value ({key}): {val:.6f}")

    if result[1] < 0.05:
        print("  => Reject H0: the series IS stationary.")
    else:
        print("  => Fail to reject H0: the series is NON-stationary.")
    print()
    return output

In [ ]:
_ = adf_test(airline["Passengers"], name="Airline Passengers")

In [ ]:
_ = adf_test(births["Births"], name="Daily Female Births")

In [ ]:
_ = adf_test(alcohol["Sales"], name="Alcohol Sales")

---
## 4. KPSS Test

The **Kwiatkowski-Phillips-Schmidt-Shin (KPSS)** test has the *opposite*
hypotheses from ADF:

- $H_0$: The series is **stationary** (trend-stationary or level-stationary)
- $H_1$: The series has a **unit root** (non-stationary)

### Interpretation

- **p-value > 0.05** $\Rightarrow$ fail to reject $H_0$ $\Rightarrow$ evidence of stationarity
- **p-value $\le$ 0.05** $\Rightarrow$ reject $H_0$ $\Rightarrow$ evidence of non-stationarity

### Why Use Both ADF and KPSS?

Using both tests together provides a more reliable conclusion:

| ADF Result | KPSS Result | Conclusion |
|------------|-------------|------------|
| Stationary | Stationary | Series is stationary |
| Non-stationary | Non-stationary | Series is non-stationary |
| Stationary | Non-stationary | Trend-stationary (difference or detrend) |
| Non-stationary | Stationary | Difference-stationary (borderline) |

In [ ]:
def kpss_test(series, name="", regression="c"):
    """Run the KPSS test and print a clear summary.

    Parameters
    ----------
    series : array-like
        The time series to test.
    name : str
        Label for display purposes.
    regression : str
        'c' for level stationarity (default), 'ct' for trend stationarity.

    Returns
    -------
    dict with test statistic, p-value, lags used, and critical values.
    """
    result = kpss(series.dropna(), regression=regression, nlags="auto")
    output = {
        "Test Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Critical Values": result[3],
    }

    label = f" ({name})" if name else ""
    print(f"KPSS Test{label}  [regression='{regression}']")
    print(f"  Test Statistic : {result[0]:.6f}")
    print(f"  p-value        : {result[1]:.6f}")
    print(f"  Lags Used      : {result[2]}")
    for key, val in result[3].items():
        print(f"  Critical Value ({key}): {val:.6f}")

    if result[1] > 0.05:
        print("  => Fail to reject H0: the series IS stationary.")
    else:
        print("  => Reject H0: the series is NON-stationary.")
    print()
    return output

In [ ]:
_ = kpss_test(airline["Passengers"], name="Airline Passengers")

In [ ]:
_ = kpss_test(births["Births"], name="Daily Female Births")

In [ ]:
_ = kpss_test(alcohol["Sales"], name="Alcohol Sales")

In [ ]:
# Summary table: ADF + KPSS combined verdict
print("Combined ADF + KPSS Results:")
print("="*60)
print(f"{'Dataset':<25} {'ADF':<15} {'KPSS':<15} {'Verdict'}")
print("-"*60)

for name, series in [("Airline Passengers", airline["Passengers"]),
                      ("Daily Female Births", births["Births"]),
                      ("Alcohol Sales", alcohol["Sales"])]:
    adf_p = adfuller(series.dropna(), autolag="AIC")[1]
    kpss_p = kpss(series.dropna(), regression="c", nlags="auto")[1]

    adf_verdict = "Stationary" if adf_p < 0.05 else "Non-stationary"
    kpss_verdict = "Stationary" if kpss_p > 0.05 else "Non-stationary"

    if adf_verdict == "Stationary" and kpss_verdict == "Stationary":
        combined = "Stationary"
    elif adf_verdict == "Non-stationary" and kpss_verdict == "Non-stationary":
        combined = "Non-stationary"
    else:
        combined = "Inconclusive"

    print(f"{name:<25} {adf_verdict:<15} {kpss_verdict:<15} {combined}")

---
## 5. Differencing to Achieve Stationarity

**Differencing** is the standard technique for removing trend and seasonality
to make a series stationary.

### First Difference

Removes a linear trend:

$$
y_t' = y_t - y_{t-1} = \nabla y_t
$$

### Seasonal Difference

Removes seasonal patterns (period $m$):

$$
y_t^* = y_t - y_{t-m} = \nabla_m y_t
$$

### Both Together

For data with both trend and seasonality, apply both:

$$
\nabla \nabla_m y_t = (y_t - y_{t-m}) - (y_{t-1} - y_{t-1-m})
$$

In [ ]:
# --- First Differencing: Airline Passengers ---
airline_d1 = airline["Passengers"].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline["Passengers"])
axes[0].set_title("Original — Airline Passengers")

axes[1].plot(airline_d1, color="tab:orange")
axes[1].set_title("First Difference — $\\nabla y_t$")

plt.tight_layout()
plt.show()

print("First differencing removes the trend, but seasonality remains.")
print("Also, the variance is still increasing — growing seasonal amplitude.")

In [ ]:
# Test the first-differenced series
print("ADF on first-differenced Airline:")
_ = adf_test(airline_d1, name="Airline diff(1)")

print("KPSS on first-differenced Airline:")
_ = kpss_test(airline_d1, name="Airline diff(1)")

In [ ]:
# --- Seasonal Differencing ---
airline_d12 = airline["Passengers"].diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline["Passengers"])
axes[0].set_title("Original")

axes[1].plot(airline_d12, color="tab:green")
axes[1].set_title("Seasonal Difference — $\\nabla_{12} y_t$")

plt.tight_layout()
plt.show()

print("Seasonal differencing removes the seasonality but a trend remains.")

In [ ]:
# --- First + Seasonal Differencing ---
airline_d1_d12 = airline["Passengers"].diff().diff(12).dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(airline["Passengers"])
axes[0, 0].set_title("Original")

axes[0, 1].plot(airline_d1, color="tab:orange")
axes[0, 1].set_title("First Difference")

axes[1, 0].plot(airline_d12, color="tab:green")
axes[1, 0].set_title("Seasonal Difference")

axes[1, 1].plot(airline_d1_d12, color="tab:red")
axes[1, 1].set_title("First + Seasonal Difference")

for ax in axes.flat:
    ax.set_xlabel("Date")

plt.suptitle("Progressive Differencing — Airline Passengers", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Test the doubly-differenced series
print("ADF on first + seasonally differenced Airline:")
_ = adf_test(airline_d1_d12, name="Airline diff(1)+diff(12)")

print("KPSS on first + seasonally differenced Airline:")
_ = kpss_test(airline_d1_d12, name="Airline diff(1)+diff(12)")

In [ ]:
# Visual confirmation: rolling stats of doubly-differenced airline
plot_rolling_stats(airline_d1_d12, window=12,
                   title="Airline (first + seasonal diff)")

print("Rolling mean and std are now approximately constant — stationarity achieved.")

### Alcohol Sales — another example

In [ ]:
# Alcohol: first + seasonal differencing
alc_d1_d12 = alcohol["Sales"].diff().diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(alcohol["Sales"])
axes[0].set_title("Alcohol Sales — Original")

axes[1].plot(alc_d1_d12, color="tab:red")
axes[1].set_title("Alcohol Sales — $\\nabla \\nabla_{12}$")

plt.tight_layout()
plt.show()

_ = adf_test(alc_d1_d12, name="Alcohol diff(1)+diff(12)")
_ = kpss_test(alc_d1_d12, name="Alcohol diff(1)+diff(12)")

---
## 6. Over-Differencing Warning

**More differencing is not always better.** Applying too many differences
introduces artificial patterns:

- Over-differencing a stationary series adds spurious negative autocorrelation.
- Each round of differencing reduces the sample by one observation.
- Over-differenced data become harder to model (MA components proliferate).

**Rule of thumb:** difference until the ADF and KPSS both agree the series is
stationary, then stop.

In [ ]:
# Demonstrate over-differencing: take the already-stationary births data
# and difference it unnecessarily
births_d1 = births["Births"].diff().dropna()
births_d2 = births_d1.diff().dropna()

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Original
axes[0, 0].plot(births["Births"])
axes[0, 0].set_title("Original — Already Stationary")
plot_acf(births["Births"], lags=30, ax=axes[0, 1], title="ACF — Original")

# One difference (unnecessary)
axes[1, 0].plot(births_d1, color="tab:orange")
axes[1, 0].set_title("First Difference — Over-differenced")
plot_acf(births_d1, lags=30, ax=axes[1, 1], title="ACF — First Diff")

# Two differences (very over-differenced)
axes[2, 0].plot(births_d2, color="tab:red")
axes[2, 0].set_title("Second Difference — Very Over-differenced")
plot_acf(births_d2, lags=30, ax=axes[2, 1], title="ACF — Second Diff")

plt.tight_layout()
plt.show()

print("Notice how each unnecessary difference adds negative autocorrelation.")
print("The original series was already stationary — differencing made it worse!")

In [ ]:
# Quantitative check: variance should increase with over-differencing
print("Variance comparison:")
print(f"  Original:        {births['Births'].var():.2f}")
print(f"  First diff:      {births_d1.var():.2f}")
print(f"  Second diff:     {births_d2.var():.2f}")
print()
print("Over-differencing inflates the variance — a telltale sign.")

---
## 7. Log Transform + Differencing

For series with **multiplicative seasonality** (seasonal amplitude grows with
the level), it is common to apply a **log transform first** and then
difference.  The log stabilises the variance, and differencing removes the
trend and seasonality.

In [ ]:
# Log + first + seasonal difference for airline
log_airline = np.log(airline["Passengers"])
log_airline_diff = log_airline.diff().diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline_d1_d12)
axes[0].set_title("Differenced (no log)")
axes[0].set_ylabel("Value")

axes[1].plot(log_airline_diff, color="tab:purple")
axes[1].set_title("Log + Differenced")
axes[1].set_ylabel("Value")

plt.tight_layout()
plt.show()

print("The log-differenced series has more constant variance —")
print("the log transform handled the multiplicative seasonality.")

In [ ]:
_ = adf_test(log_airline_diff, name="log(Airline) diff(1)+diff(12)")
_ = kpss_test(log_airline_diff, name="log(Airline) diff(1)+diff(12)")

---
## Summary

| Concept | Key Idea |
|---------|----------|
| Weak stationarity | Constant mean, constant variance, autocovariance depends only on lag |
| Visual test | Rolling mean and rolling std should be approximately constant |
| ADF test | H0: non-stationary; p < 0.05 $\Rightarrow$ stationary |
| KPSS test | H0: stationary; p > 0.05 $\Rightarrow$ stationary |
| First difference | $\nabla y_t = y_t - y_{t-1}$ removes linear trend |
| Seasonal difference | $\nabla_m y_t = y_t - y_{t-m}$ removes seasonality |
| Over-differencing | Inflates variance, adds spurious negative autocorrelation — stop once stationary |
| Log + diff | For multiplicative seasonality: log first, then difference |

In [ ]:
print("Next: Notebook 03 — Statistical Tests for Time Series")
print("We will cover Ljung-Box, Granger causality, and cointegration tests.")